# 41 Scene Training (YOLOE experiment, retained)

**Purpose:** Keep the YOLOE + RF-DETR scene-feature experiment for comparison with the CLIP branch that production uses.

**Inputs:** Two `scene_objects` train images, YOLOE `models/scene/yoloe-11s-seg-pf.pt` and RF-DETR weights (downloaded on first use).

**Outputs:** A temporary demonstration cache only (deleted at the end of the cell); no project files are written.

**Splits used:** train only (two example images); test: not used.

Production uses **CLIP**. See the [CLIP handoff guide](../docs/scene_fusion.md)
for the active segment prediction and export workflow. This notebook preserves the
YOLOE + RF-DETR experiment, including local copies of its encoder and v2 cache
helpers. Its caches are incompatible with the active CLIP v3 export format.


Frozen prompt-free YOLOE extracts environmental objects, and the project's RF-DETR dog detector supplies dog boxes from the same RGB frames. A small feature-pooling class produces one fixed-length float32 vector per bark segment. Fusion learns the behavioral categories from these scene features alongside audio and body features.

There is no scene classifier or training loop here. The classes follow the coursework style: settings in `__init__` and explicit computations in small methods. Legacy encoder and cache definitions are preserved in this notebook. The active application scene branch uses CLIP.

In [ ]:
import hashlib
import json
import tempfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.patches import Rectangle
from PIL import Image

from woofdata.loader import load_sample
from woofdata.path import load_data_dir_from_env_file
from woofdata.tasks import task_rows
from woofdecoder.config import REPO_ROOT, load_config
from woofdecoder.preprocess.dog_detector import detect_dogs, load_dog_detector
from woofdecoder.preprocess.segments import BarkSegment
from woofdecoder.taxonomy import CATEGORIES

pd.set_option("display.width", 200)

In [ ]:
load_data_dir_from_env_file()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIG = load_config()
# This notebook owns its legacy settings; production branches.scene stays CLIP.
CONFIG["branches"]["scene"] = {
    "encoder": "yoloe_rfdetr_object_pool_v2",
    "model": "models/scene/yoloe-11s-seg-pf.pt",
    "confidence": 0.6,
    "image_size": 640,
    "max_objects": 32,
    "dog_overlap_iou": 0.5,
}

## 1. Setup

If the selected kernel is missing packages, run `%pip install torch ultralytics rfdetr numpy pandas pillow matplotlib`. Use the project's Python 3.12 environment. YOLOE weights are loaded from `models/scene/yoloe-11s-seg-pf.pt` and downloaded there on first use if absent.

The preprocessing team supplies full RGB frames. The scene encoder uses them directly, including frames where no dog appears.

Run cells in order. Legacy encoder and cache helpers below are local to this notebook; its explicit YOLOE settings do not change the production CLIP configuration.

RF-DETR uses the existing dog_detector settings (Nano, confidence 0.5). YOLOE keeps scene confidence 0.6. The dog detector downloads its pretrained weights on first use. An existing preprocessing DogDetector can be shared with encoder_from_config(CONFIG, dog_detector=detector) to avoid loading it twice.

In [ ]:
print("Device:", DEVICE)

## Preserved experimental implementation
These definitions are notebook-local and are never imported by the CLIP branch.

In [ ]:
"""Frozen YOLOE objects and RF-DETR dogs for multimodal fusion."""

DEFAULT_CHECKPOINT = REPO_ROOT / "models/scene/yoloe-11s-seg-pf.pt"


def box_iou(first, second):
    x1 = max(first[0], second[0])
    y1 = max(first[1], second[1])
    x2 = min(first[2], second[2])
    y2 = min(first[3], second[3])
    intersection = max(x2 - x1, 0) * max(y2 - y1, 0)
    first_area = max(first[2] - first[0], 0) * max(first[3] - first[1], 0)
    second_area = max(second[2] - second[0], 0) * max(second[3] - second[1], 0)
    union = first_area + second_area - intersection
    return float(intersection / union) if union > 0 else 0.0


def merge_dog_detections(
    yoloe_objects, dogs, frame_index, frame_shape, dog_class_id, overlap_iou=0.5
):
    """Prefer RF-DETR's dog label when both detectors describe the same box."""
    if not 0 < overlap_iou <= 1:
        raise ValueError("overlap_iou must be between 0 and 1")
    if not isinstance(frame_index, int) or frame_index < 0:
        raise ValueError("frame_index must be a nonnegative integer")
    if not isinstance(dog_class_id, int) or dog_class_id < 0:
        raise ValueError("dog_class_id must be a nonnegative integer")
    height, width = frame_shape[:2]
    if height <= 0 or width <= 0:
        raise ValueError("Frame dimensions must be positive")
    dog_objects = []
    for box, confidence in dogs:
        values = np.asarray(box, dtype=float)
        if values.shape != (4,) or not np.isfinite(values).all():
            raise ValueError("Dog boxes must be finite pixel xywh coordinates")
        if not np.isfinite(confidence) or not 0 < confidence <= 1:
            raise ValueError("Dog confidence must be between 0 and 1")
        x, y, box_width, box_height = values
        x1, x2 = np.clip([x, x + box_width], 0, width) / width
        y1, y2 = np.clip([y, y + box_height], 0, height) / height
        if x2 <= x1 or y2 <= y1:
            continue
        dog_objects.append(
            {
                "frame": frame_index,
                "class_id": dog_class_id,
                "name": "dog",
                "confidence": float(confidence),
                "box": [float(x1), float(y1), float(x2), float(y2)],
                "source": "rfdetr",
            }
        )
    kept, suppressed = [], []
    for obj in yoloe_objects:
        overlap = max((box_iou(obj["box"], dog["box"]) for dog in dog_objects), default=0.0)
        if obj["frame"] == frame_index and overlap >= overlap_iou:
            suppressed.append({**obj, "matched_dog_iou": overlap})
        else:
            kept.append(obj)
    return kept + dog_objects, suppressed


class ObjectFeaturePool:
    feature_names = (
        "frame_fraction",
        "mean_count",
        "max_confidence",
        "mean_center_x",
        "mean_center_y",
        "mean_width",
        "mean_height",
        "mean_area",
        "delta_center_x",
        "delta_center_y",
    )

    def __init__(self, names):
        self.names = tuple(names)
        if not self.names or len(set(self.names)) != len(self.names):
            raise ValueError("Use a nonempty, unique detector vocabulary")
        self.dim = len(self.names) * len(self.feature_names)

    def encode(self, detections, frame_count):
        if not isinstance(frame_count, int) or frame_count < 0:
            raise ValueError("frame_count must be a nonnegative integer")
        counts = np.zeros((frame_count, len(self.names)), dtype=np.float32)
        scores = np.zeros_like(counts)
        weight_sums = np.zeros_like(counts)
        geometry = np.zeros((frame_count, len(self.names), 5), dtype=np.float32)

        for obj in detections:
            frame, class_id = obj["frame"], obj["class_id"]
            if not isinstance(frame, int) or not 0 <= frame < frame_count:
                raise ValueError("Detection frame is outside this segment")
            if not isinstance(class_id, int) or not 0 <= class_id < len(self.names):
                raise ValueError("Detection class ID is outside the saved vocabulary")
            if obj["name"] != self.names[class_id]:
                raise ValueError("Detection name and class ID disagree with the vocabulary")
            x1, y1, x2, y2 = obj["box"]
            confidence = obj["confidence"]
            values = np.array([x1, y1, x2, y2, confidence])
            if not np.isfinite(values).all() or not ((values >= 0) & (values <= 1)).all():
                raise ValueError("Boxes and confidence must be normalized between 0 and 1")
            if x2 <= x1 or y2 <= y1 or confidence <= 0:
                raise ValueError("Detections need positive box area and confidence")

            width, height = x2 - x1, y2 - y1
            box_features = np.array(
                [
                    (x1 + x2) / 2,
                    (y1 + y2) / 2,
                    width,
                    height,
                    width * height,
                ]
            )
            counts[frame, class_id] += 1
            scores[frame, class_id] = max(scores[frame, class_id], confidence)
            geometry[frame, class_id] += confidence * box_features
            weight_sums[frame, class_id] += confidence

        pooled = np.zeros((len(self.names), len(self.feature_names)), dtype=np.float32)
        if frame_count == 0:
            return pooled.reshape(-1)

        geometry /= np.maximum(weight_sums[:, :, None], np.finfo(np.float32).tiny)
        pooled[:, 0] = (counts > 0).mean(axis=0)
        pooled[:, 1] = counts.mean(axis=0)
        pooled[:, 2] = scores.max(axis=0)
        for class_id in np.flatnonzero(counts.sum(axis=0)):
            observed = np.flatnonzero(counts[:, class_id])
            positions = geometry[observed, class_id]
            pooled[class_id, 3:8] = positions.mean(axis=0)
            pooled[class_id, 8:10] = positions[-1, :2] - positions[0, :2]
        return pooled.reshape(-1)


class SceneEncoder:
    name = "yoloe_rfdetr_object_pool_v2"

    def __init__(
        self,
        checkpoint=DEFAULT_CHECKPOINT,
        device=None,
        confidence=0.25,
        image_size=640,
        max_objects=32,
        dog_detector=None,
        dog_overlap_iou=0.5,
    ):
        from ultralytics import YOLOE

        checkpoint = Path(checkpoint)
        if not checkpoint.is_absolute():
            checkpoint = REPO_ROOT / checkpoint
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        if not checkpoint.name.endswith("-seg-pf.pt"):
            raise ValueError("Use a prompt-free YOLOE checkpoint ending in -seg-pf.pt")
        if not 0 < confidence <= 1 or image_size < 1 or max_objects < 1:
            raise ValueError("Confidence, image size, and detection limit must be positive")
        if not 0 < dog_overlap_iou <= 1:
            raise ValueError("dog_overlap_iou must be between 0 and 1")
        self.device = device
        self.confidence = confidence
        self.image_size = image_size
        self.max_objects = max_objects
        self.model = YOLOE(str(checkpoint))
        self.model.model.eval()
        for parameter in self.model.model.parameters():
            parameter.requires_grad_(False)
        names = self.model.names
        self.pool = ObjectFeaturePool([names[i] for i in range(len(names))])
        self.names = self.pool.names
        self.dim = self.pool.dim
        if "dog" not in self.names:
            raise ValueError("The YOLOE vocabulary must include the canonical dog class")
        self.dog_class_id = self.names.index("dog")
        self.dog_detector = (
            dog_detector
            if dog_detector is not None
            else load_dog_detector(verbose=False, device=device)
        )
        self.dog_overlap_iou = dog_overlap_iou
        try:
            dog_version = version("rfdetr")
        except PackageNotFoundError:
            dog_version = None
        dog_schema = {
            "model": type(self.dog_detector.model).__name__,
            "package_version": dog_version,
            "confidence": self.dog_detector.score_threshold,
            "overlap_iou": dog_overlap_iou,
            "merge_policy": "prefer_dog_for_high_iou_boxes_v1",
        }
        dog_config = getattr(self.dog_detector.model, "model_config", None)
        dog_checkpoint = getattr(dog_config, "pretrain_weights", None)
        if dog_checkpoint and Path(dog_checkpoint).is_file():
            dog_path = Path(dog_checkpoint)
            dog_schema["checkpoint"] = dog_path.name
            dog_schema["checkpoint_sha256"] = hashlib.sha256(dog_path.read_bytes()).hexdigest()
        self.schema = {
            "encoder": self.name,
            "checkpoint": checkpoint.name,
            "checkpoint_sha256": hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
            "vocabulary": list(self.names),
            "feature_names": list(self.pool.feature_names),
            "layout": "class_major",
            "confidence": confidence,
            "image_size": image_size,
            "max_yoloe_objects_per_frame": max_objects,
            "dog_detector": dog_schema,
        }
        schema_text = json.dumps(self.schema, sort_keys=True)
        self.schema_id = hashlib.sha256(schema_text.encode()).hexdigest()

    @torch.inference_mode()
    def encode_frames(self, frames, frame_times_s=None):
        frames = list(frames)
        if frame_times_s is not None:
            times = np.asarray(frame_times_s, dtype=float)
            if times.shape != (len(frames),) or not np.isfinite(times).all():
                raise ValueError("Supply one finite timestamp per frame")
            if (np.diff(times) < 0).any():
                raise ValueError("Frame timestamps must be in chronological order")
        detections, yoloe_detections, dog_detections, suppressed = [], [], [], []
        self.model.model.eval()
        for frame_index, frame in enumerate(frames):
            image = np.asarray(frame)
            if image.ndim != 3 or image.shape[-1] != 3 or image.dtype != np.uint8:
                raise ValueError("Frames must be uint8 RGB images")
            result = self.model.predict(
                source=Image.fromarray(image),
                device=self.device,
                conf=self.confidence,
                imgsz=self.image_size,
                max_det=self.max_objects,
                verbose=False,
            )[0]
            names = tuple(result.names[i] for i in range(len(result.names)))
            if names != self.names:
                raise ValueError("Detector vocabulary changed")
            frame_objects = []
            for box in result.boxes:
                class_id = int(box.cls.item())
                detection = {
                    "frame": frame_index,
                    "class_id": class_id,
                    "name": self.names[class_id],
                    "confidence": float(box.conf.item()),
                    "box": box.xyxyn[0].cpu().tolist(),
                    "source": "yoloe",
                }
                frame_objects.append(detection)
            # Run one frame at a time; keep every dog returned by preprocessing's detector.
            dogs = detect_dogs(self.dog_detector, [image.copy()])[0]
            combined, replaced = merge_dog_detections(
                frame_objects,
                dogs,
                frame_index,
                image.shape,
                self.dog_class_id,
                self.dog_overlap_iou,
            )
            if frame_times_s is not None:
                for detection in frame_objects + combined + replaced:
                    detection["time_s"] = float(times[frame_index])
            detections.extend(combined)
            yoloe_detections.extend(frame_objects)
            dog_detections.extend(obj for obj in combined if obj["source"] == "rfdetr")
            suppressed.extend(replaced)
        return {
            "embedding": self.pool.encode(detections, len(frames)),
            "detected_objects": detections,
            "yoloe_objects": yoloe_detections,
            "dog_objects": dog_detections,
            "suppressed_objects": suppressed,
            "present": len(frames) > 0,
            "has_detections": bool(detections),
            "frame_count": len(frames),
            "schema_id": self.schema_id,
        }

    def embed_frames(self, frames):
        return self.encode_frames(frames)["embedding"]

    def encode_segment(self, segment: BarkSegment):
        """Return scene features and the source interval used by preprocessing."""
        times = np.asarray(segment.frame_times_s, dtype=float)
        if not np.isfinite([segment.start_s, segment.end_s]).all():
            raise ValueError("Segment boundaries must be finite")
        if times.size and (
            (times < segment.start_s - 1e-3).any() or (times > segment.end_s + 1e-3).any()
        ):
            raise ValueError("Frame timestamps must fall inside the segment")
        result = self.encode_frames(segment.frames, segment.frame_times_s)
        result.update(
            source_id=segment.source_id,
            start_s=float(segment.start_s),
            end_s=float(segment.end_s),
            frame_times_s=times.tolist(),
        )
        return result

    def encode_segments(self, segments):
        return [self.encode_segment(segment) for segment in segments]

    def embed(self, segment: BarkSegment) -> np.ndarray | None:
        if not segment.has_video:
            return None
        return self.encode_segment(segment)["embedding"]


def encoder_from_config(config: dict, device: str | None = None, dog_detector=None) -> SceneEncoder:
    """Load the scene encoder using branches.scene settings."""
    section = config["branches"]["scene"]
    if section.get("encoder") != SceneEncoder.name:
        raise ValueError(f"Unsupported scene encoder: {section.get('encoder')!r}")
    if not section.get("model"):
        raise ValueError("Set branches.scene.model in configs/pipeline.yaml")
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if dog_detector is None:
        dog_section = config["dog_detector"]
        dog_detector = load_dog_detector(
            model_size=dog_section["model"],
            score_threshold=float(dog_section["score_threshold"]),
            verbose=False,
            device=device,
        )
    return SceneEncoder(
        checkpoint=section["model"],
        device=device,
        confidence=float(section.get("confidence", 0.25)),
        image_size=int(section.get("image_size", 640)),
        max_objects=int(section.get("max_objects", 32)),
        dog_detector=dog_detector,
        dog_overlap_iou=float(section.get("dog_overlap_iou", 0.5)),
    )

In [ ]:
"""Export aligned scene features; reading caches does not load either detector."""

CACHE_FORMAT_VERSION = 2
SEGMENT_FIELDS = ("source_id", "start_s", "end_s", "frame_times_s")
RESERVED_FIELDS = {"clip_id", "present", "has_detections", "frame_count"}


def schema_identity(schema):
    return hashlib.sha256(json.dumps(schema, sort_keys=True).encode()).hexdigest()


def check_ids(clip_ids):
    clip_ids = list(clip_ids)
    if not clip_ids or any(not isinstance(value, str) or not value.strip() for value in clip_ids):
        raise ValueError("Segment IDs must be nonempty strings")
    if len(set(clip_ids)) != len(clip_ids):
        raise ValueError("Segment IDs must be unique")
    return clip_ids


def schema_dimension(schema):
    for key in ("vocabulary", "feature_names"):
        values = schema.get(key)
        if (
            not isinstance(values, list)
            or not values
            or any(not isinstance(v, str) for v in values)
        ):
            raise ValueError(f"Invalid scene schema {key}")
        if len(set(values)) != len(values):
            raise ValueError(f"Duplicate entries in scene schema {key}")
    return len(schema["vocabulary"]) * len(schema["feature_names"])


def check_metadata(metadata, frame_count):
    if metadata.get("mapped_category") not in (None, "", *CATEGORIES):
        raise ValueError("mapped_category must use the shared taxonomy or be unannotated")
    if metadata.get("split") not in (None, "", "train", "val", "test", "train_only"):
        raise ValueError("Unknown dataset split")
    start, end = metadata.get("start_s"), metadata.get("end_s")
    if (start is None) != (end is None):
        raise ValueError("Supply both segment boundaries or neither")
    if start is not None and (not np.isfinite([start, end]).all() or end < start):
        raise ValueError("Invalid segment boundaries")
    times = metadata.get("frame_times_s")
    if times is not None:
        times = np.asarray(times, dtype=float)
        if (
            times.shape != (frame_count,)
            or not np.isfinite(times).all()
            or (np.diff(times) < 0).any()
        ):
            raise ValueError("Invalid frame timestamps")
        if start is not None and ((times < start - 1e-3).any() or (times > end + 1e-3).any()):
            raise ValueError("Frame timestamps lie outside their segment")


def check_result(result, embedding):
    count = result["frame_count"]
    if isinstance(count, bool) or not isinstance(count, (int, np.integer)) or count < 0:
        raise ValueError("frame_count must be a nonnegative integer")
    for key in ("present", "has_detections"):
        if not isinstance(result[key], (bool, np.bool_)):
            raise TypeError(f"{key} must be a boolean")
    if result["present"] != (count > 0):
        raise ValueError("Availability mask disagrees with frame_count")
    if result["has_detections"] != bool(result["detected_objects"]):
        raise ValueError("Detection mask disagrees with detected_objects")
    if not result["present"] and result["has_detections"]:
        raise ValueError("Missing video cannot have detections")
    if not result["has_detections"] and np.any(embedding):
        raise ValueError("Empty scene detections must have a zero embedding")


def save_scene_cache(path, clip_ids, results, encoder, metadata=None):
    """Save one row per shared segment ID, including optional genuine annotations."""
    clip_ids = check_ids(clip_ids)
    results = list(results)
    if len(clip_ids) != len(results):
        raise ValueError("Supply one segment ID per encoder result")
    dimension = schema_dimension(encoder.schema)
    if dimension != encoder.dim or schema_identity(encoder.schema) != encoder.schema_id:
        raise ValueError("Encoder schema identity or dimension is inconsistent")
    if any(result["schema_id"] != encoder.schema_id for result in results):
        raise ValueError("Encoder result schema differs from the cache schema")
    embeddings = np.stack([result["embedding"] for result in results]).astype(np.float32)
    if embeddings.shape != (len(clip_ids), dimension) or not np.isfinite(embeddings).all():
        raise ValueError("Invalid scene embedding shape or values")
    metadata = [{} for _ in results] if metadata is None else list(metadata)
    if len(metadata) != len(results):
        raise ValueError("Supply one metadata record per segment")
    records = []
    for result, embedding, extra in zip(results, embeddings, metadata, strict=True):
        check_result(result, embedding)
        if not isinstance(extra, dict) or RESERVED_FIELDS.intersection(extra):
            raise ValueError("Metadata cannot override segment IDs or availability fields")
        record = {key: result.get(key) for key in SEGMENT_FIELDS}
        for key, value in extra.items():
            if key in SEGMENT_FIELDS and record[key] is not None and record[key] != value:
                raise ValueError(f"Metadata disagrees with encoded segment {key}")
            record[key] = value
        check_metadata(record, result["frame_count"])
        records.append(record)
    metadata_json = np.array([json.dumps(record, allow_nan=False) for record in records], dtype=str)
    detections_json = np.array(
        [json.dumps(result["detected_objects"], allow_nan=False) for result in results], dtype=str
    )
    path = Path(path)
    if path.suffix.lower() != ".npz":
        raise ValueError("Scene cache path must end in .npz")
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".npz", delete=False) as stream:
            temporary = Path(stream.name)
            np.savez_compressed(
                stream,
                format_version=np.array(CACHE_FORMAT_VERSION),
                clip_id=np.array(clip_ids, dtype=str),
                embedding=embeddings,
                present=np.array([result["present"] for result in results], dtype=bool),
                has_detections=np.array(
                    [result["has_detections"] for result in results], dtype=bool
                ),
                frame_count=np.array([result["frame_count"] for result in results], dtype=np.int32),
                schema_json=np.array(json.dumps(encoder.schema, sort_keys=True)),
                schema_id=np.array(encoder.schema_id),
                metadata_json=metadata_json,
                detections_json=detections_json,
            )
        temporary.replace(path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
    return path


def cache_scene_segments(path, clip_ids, segments, encoder, metadata=None):
    """Encode the preprocessing team's segments without decoding videos again."""
    clip_ids = check_ids(clip_ids)
    results = [
        encoder.encode_segment(segment) for _, segment in zip(clip_ids, segments, strict=True)
    ]
    return save_scene_cache(path, clip_ids, results, encoder, metadata)


def load_scene_cache(path, encoder=None, clip_ids=None, expected_schema_id=None):
    """Read features without model weights; optionally align them to another modality's IDs."""
    with np.load(path, allow_pickle=False) as cached:
        if (
            "format_version" not in cached
            or cached["format_version"].item() != CACHE_FORMAT_VERSION
        ):
            raise ValueError("Scene cache format is outdated; rebuild the cache")
        schema = json.loads(cached["schema_json"].item())
        if encoder is not None and schema != encoder.schema:
            raise ValueError("Cached vocabulary, feature order, or detector settings differ")
        schema_id = schema_identity(schema)
        if cached["schema_id"].item() != schema_id:
            raise ValueError("Cached scene schema hash is invalid")
        if expected_schema_id is not None and expected_schema_id != schema_id:
            raise ValueError("Cached scene schema does not match the expected schema ID")
        ids = check_ids(cached["clip_id"].tolist())
        embeddings = cached["embedding"].astype(np.float32)
        if (
            embeddings.shape != (len(ids), schema_dimension(schema))
            or not np.isfinite(embeddings).all()
        ):
            raise ValueError("Cached scene embedding shape or values are invalid")
        columns = {key: cached[key] for key in ("present", "has_detections", "frame_count")}
        if any(values.shape != (len(ids),) for values in columns.values()):
            raise ValueError("Cached availability columns have invalid shapes")
        if cached["metadata_json"].shape != (len(ids),) or cached["detections_json"].shape != (
            len(ids),
        ):
            raise ValueError("Cached metadata or detections have invalid shapes")
        records = []
        for i, clip_id in enumerate(ids):
            metadata = json.loads(cached["metadata_json"][i])
            if not isinstance(metadata, dict) or RESERVED_FIELDS.intersection(metadata):
                raise ValueError("Cached metadata overrides segment identifiers or masks")
            result = {key: values[i] for key, values in columns.items()}
            result["detected_objects"] = json.loads(cached["detections_json"][i])
            if not isinstance(result["detected_objects"], list):
                raise TypeError("Cached detections must be a list")
            check_result(result, embeddings[i])
            check_metadata(metadata, result["frame_count"])
            records.append(
                {
                    "clip_id": clip_id,
                    **metadata,
                    **{key: values[i] for key, values in columns.items()},
                }
            )
    table = pd.DataFrame(records)
    if clip_ids is not None:
        wanted = check_ids(clip_ids)
        index = pd.Index(ids).get_indexer(wanted)
        if (index < 0).any():
            raise ValueError("Requested segment IDs are missing from the scene cache")
        embeddings = embeddings[index]
        table = table.iloc[index].reset_index(drop=True)
    table.attrs["scene_schema"] = schema
    table.attrs["scene_schema_id"] = schema_id
    return embeddings, table

## 2. Fixed-length object features

Each detector class gets ten features in the order below. The full vector flattens the class-by-feature table, so its shape stays fixed when frame or object counts change.

| Feature | Calculation |
|---|---|
| Frame fraction | Fraction of frames containing this class |
| Mean count | Number of detections per frame, averaged across all frames |
| Maximum confidence | Highest detector confidence across the segment |
| Mean center x/y, width, height, area | Confidence-weighted within each frame, then averaged over frames where the class appears |
| Delta center x/y | Last observed class center minus first observed class center |

Positions and sizes are normalized to the full frame. Deltas compare class centers; they do not track individual objects or measure speed. Missing classes have zero features. Pooling loses detailed object-to-object relationships, so it provides a simple starting point for fusion.

## 3. Frozen encoder

YOLOE prompt-free uses its complete built-in vocabulary without text prompts or a hand-picked object list. RGB images are passed through PIL to preserve channel order. Frames run one at a time to limit GPU memory.

`encode_frames(frames)` returns the vector plus detections and availability flags. `embed(segment)` follows the audio encoder's segment interface and returns the vector, or `None` when video is missing. This is a feature interface for fusion; detector confidence is not a behavioral probability.

For each RF-DETR dog, YOLOE boxes in the same frame with IoU at least 0.5 are replaced by the canonical dog detection. Other YOLOE boxes stay, including small objects inside a dog box. All dogs are retained, rather than only the largest dog. Detections include their source, and raw/suppressed YOLOE results are available for comparison. This spatial matching rule can also replace a real overlapping object if RF-DETR is wrong; both detectors need evaluation.

## 4. Try two real images

These are two different recording groups from the prepared rf_leash training split. They demonstrate feature extraction only. The dataset's object labels are not used as encoder inputs, training targets, or barking categories. A single image is treated as a one-frame example.

In [ ]:
scene_encoder = encoder_from_config(CONFIG, device=DEVICE)
print("Object vocabulary:", len(scene_encoder.names))
print("Features per class:", len(scene_encoder.pool.feature_names))
print("Scene vector dimension:", scene_encoder.dim)
print("Float32 storage per vector:", scene_encoder.dim * 4, "bytes")
print(
    "Trainable detector parameters:",
    sum(p.numel() for p in scene_encoder.model.model.parameters() if p.requires_grad),
)

rows = task_rows("scene_objects", "train")
rows = rows[rows["source"].eq("rf_leash")].sort_values("clip_id")
rows = rows.drop_duplicates("leakage_group").head(2)
if len(rows) != 2:
    raise RuntimeError("Prepare at least two different rf_leash recording groups")
samples = [load_sample("scene_objects", row) for _, row in rows.iterrows()]
clip_ids = [sample.clip_id for sample in samples]
scene_results = [scene_encoder.encode_frames([sample.image]) for sample in samples]
scene_x = np.stack([result["embedding"] for result in scene_results])

print("Batch shape:", scene_x.shape)
print("Finite features:", np.isfinite(scene_x).all())
for clip_id, result in zip(clip_ids, scene_results, strict=True):
    print(clip_id)
    print(
        "YOLOE objects:",
        len(result["yoloe_objects"]),
        "| RF-DETR dogs:",
        len(result["dog_objects"]),
        "| Replaced labels:",
        [obj["name"] for obj in result["suppressed_objects"]],
    )
    display(pd.DataFrame(result["detected_objects"]).head(12))

In [ ]:
fig, axes = plt.subplots(len(samples), 2, figsize=(12, 5 * len(samples)))
for row, (sample, result) in enumerate(zip(samples, scene_results, strict=True)):
    height, width = sample.image.shape[:2]
    views = [
        ("YOLOE only", result["yoloe_objects"]),
        ("YOLOE + RF-DETR dogs", result["detected_objects"]),
    ]
    for column, (title, objects) in enumerate(views):
        axis = axes[row, column]
        axis.imshow(sample.image)
        for obj in objects:
            x1, y1, x2, y2 = obj["box"]
            color = "orange" if obj["source"] == "rfdetr" else "lime"
            axis.add_patch(
                Rectangle(
                    (x1 * width, y1 * height),
                    (x2 - x1) * width,
                    (y2 - y1) * height,
                    fill=False,
                    edgecolor=color,
                    linewidth=1,
                )
            )
            axis.text(
                x1 * width,
                y1 * height,
                f"{obj['name']} {obj['confidence']:.2f}",
                fontsize=7,
                color="black",
                backgroundcolor=color,
            )
        axis.set_title(title)
        axis.axis("off")
fig.tight_layout()
display(fig)
plt.close(fig)

for clip_id, result in zip(clip_ids, scene_results, strict=True):
    features = pd.DataFrame(
        result["embedding"].reshape(len(scene_encoder.names), -1),
        index=scene_encoder.names,
        columns=scene_encoder.pool.feature_names,
    )
    print("Nonzero object features:", clip_id)
    display(features[features["frame_fraction"] > 0].sort_values("max_confidence", ascending=False))

## 5. Check the encoder contract

Small mock detections verify the pooling math, empty scenes, vocabulary checks, and temporal ordering. The real-image checks verify fixed dimensions, RGB handling, repeatability, missing-video behavior, and a multi-frame BarkSegment. No optimization or behavioral evaluation is performed.

In [ ]:
def check_feature_pool():
    pool = ObjectFeaturePool(["person", "ball"])
    objects = [
        {
            "frame": 0,
            "class_id": 0,
            "name": "person",
            "confidence": 0.5,
            "box": [0.0, 0.0, 0.2, 0.4],
        },
        {
            "frame": 1,
            "class_id": 0,
            "name": "person",
            "confidence": 0.9,
            "box": [0.4, 0.2, 0.6, 0.6],
        },
    ]
    values = pool.encode(objects, 3).reshape(2, 10)
    expected = [2 / 3, 2 / 3, 0.9, 0.3, 0.3, 0.2, 0.4, 0.08, 0.4, 0.2]
    np.testing.assert_allclose(values[0], expected, atol=1e-6)
    assert not values[1].any()
    assert not pool.encode([], 3).any()
    assert not pool.encode([], 0).any()
    np.testing.assert_allclose(pool.encode(objects[::-1], 3), values.reshape(-1))
    reversed_frames = [{**obj, "frame": 1 - obj["frame"]} for obj in objects]
    reversed_values = pool.encode(reversed_frames, 3).reshape(2, 10)
    np.testing.assert_allclose(reversed_values[:, :8], values[:, :8], atol=1e-6)
    np.testing.assert_allclose(reversed_values[:, 8:], -values[:, 8:], atol=1e-6)
    multiple = [objects[0], {**objects[0], "box": [0.4, 0.0, 0.6, 0.4]}]
    same_frame = pool.encode(multiple, 1).reshape(2, 10)[0]
    np.testing.assert_allclose(same_frame[[0, 1, 3]], [1, 2, 0.3], atol=1e-6)
    invalid = [{**objects[0], "name": "wrong"}]
    try:
        pool.encode(invalid, 3)
    except ValueError:
        pass
    else:
        raise AssertionError("Vocabulary mismatches must fail")


check_feature_pool()
assert scene_x.shape == (2, scene_encoder.dim)
assert scene_x.dtype == np.float32 and np.isfinite(scene_x).all()
assert not np.allclose(scene_x[0], scene_x[1])
assert not any(p.requires_grad for p in scene_encoder.model.model.parameters())
repeat = scene_encoder.embed_frames([samples[0].image])
np.testing.assert_allclose(repeat, scene_x[0], atol=1e-5)
missing = scene_encoder.encode_frames([])
assert not missing["present"] and not missing["has_detections"]
assert not missing["embedding"].any()
assert scene_encoder.embed(BarkSegment(0, 1)) is None
segment = BarkSegment(
    start_s=0,
    end_s=1,
    frames=[samples[0].image, samples[1].image],
    frame_times_s=[0.0, 1.0],
    source_id="two-image-interface-check",
)
segment_result = scene_encoder.encode_segment(segment)
segment_embedding = scene_encoder.embed(segment)
np.testing.assert_allclose(segment_embedding, segment_result["embedding"], atol=1e-5)
assert segment_embedding.shape == (scene_encoder.dim,)
assert segment_result["frame_count"] == 2
assert all("time_s" in obj for obj in segment_result["detected_objects"])
print("Pooling, real-image, repeatability, missing-video, and BarkSegment checks passed.")
print("Multi-frame vector:", segment_embedding.shape)
print("Two-image example checks the interface; its images are not a real bark sequence.")
assert segment_result["source_id"] == segment.source_id
assert segment_result["start_s"] == segment.start_s
assert segment_result["end_s"] == segment.end_s
assert segment_result["frame_times_s"] == segment.frame_times_s

## 6. Cache features for fusion

Cache CPU float32 vectors under the same segment IDs used for audio and body. The cache stores the complete class-ID vocabulary, feature order, detector weight hash, and inference settings. Loading rejects a different schema. A zero vector with `present=False` marks missing video; video with no detected objects is zero with `present=True`.

The demonstration uses a temporary cache, leaving project data unchanged. For a real extraction run, supply your chosen cache path and results for the team's already-preprocessed segments.

The hybrid schema also records the RF-DETR model, package/checkpoint identity, threshold, and overlap rule. Rebuild older YOLOE-only caches even though the feature-vector dimensions remain the same.

The version 2 export adds segment source/interval metadata and strict ID/mask validation. Files are replaced atomically. The fusion teammate can call load_scene_cache(path) without loading model weights, and request rows in the same explicit ID order as audio/body features.

In [ ]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "scene_embeddings.npz"
    save_scene_cache(path, clip_ids, scene_results, scene_encoder)
    cached_x, cached_rows = load_scene_cache(path)
    np.testing.assert_array_equal(cached_x, scene_x)
    assert cached_rows["clip_id"].tolist() == clip_ids
    aligned_x, aligned_rows = load_scene_cache(
        path, clip_ids=clip_ids[::-1], expected_schema_id=scene_encoder.schema_id
    )
    np.testing.assert_array_equal(aligned_x, cached_x[::-1])
    assert aligned_rows["clip_id"].tolist() == clip_ids[::-1]
    assert cached_rows.attrs["scene_schema_id"] == scene_encoder.schema_id
    schema = scene_encoder.schema
    scene_encoder.schema = {**schema, "vocabulary": list(reversed(schema["vocabulary"]))}
    try:
        load_scene_cache(path, scene_encoder)
    except ValueError:
        print("Mismatched vocabulary correctly rejected.")
    else:
        raise AssertionError("Cache schema mismatch must fail")
    finally:
        scene_encoder.schema = schema
print("Cache round trip passed:", cached_x.shape)
display(cached_rows)

## 7. Historical feature handoff

The local helpers above preserve the original YOLOE v2 cache experiment. Use them
only for this comparison, with a separate cache filename. The production exporter
`python -m wooftrainer.scene_trainer.export` now runs CLIP and writes v3 caches, so it does
not export this notebook's YOLOE encoder.

For the active five-probability segment handoff, use the
[CLIP handoff guide](../docs/scene_fusion.md). The example images here are not a
labeled barking dataset.

In [ ]:
scene_mask = cached_rows["present"].to_numpy(dtype=bool)
object_mask = cached_rows["has_detections"].to_numpy(dtype=bool)
print("Scene features:", cached_x.shape, cached_x.dtype)
print("Video availability:", scene_mask.tolist())
print("Detected-object availability:", object_mask.tolist())
print("Scene schema:", cached_rows.attrs["scene_schema_id"])
print("Align other modalities with these clip_ids:", cached_rows["clip_id"].tolist())

In [ ]:
result = scene_encoder.encode_frames([samples[0].image])

print(result["embedding"].shape)  # (45850,)
display(pd.DataFrame(result["detected_objects"]))

## Results and decisions

Production uses **CLIP** (see the introduction and [scene_fusion.md](../docs/scene_fusion.md)); this YOLOE experiment is kept for comparison only.

**Limits**

YOLOE can miss objects, confuse similar names, or split one concept across several vocabulary entries. Confidence is an object-detection score. Count, position, and change features do not establish why the dog barked. Pooling discards individual relationships and fine motion. Empty detections do not prove the environment is empty.

This notebook requires no behavioral or trigger labels to extract features. The later fusion model needs labeled, aligned examples to learn behavioral predictions. Keep dog/recording groups separate when creating its train, validation, and test sets.

RF-DETR dog repair only affects boxes matching its dog detections; it does not fix YOLOE's errors on other objects. Dog boxes and scores come from RF-DETR, not a renamed high-confidence YOLOE prediction.